# L2 – Person 1: XGBoost Multiclass Classification
**Dataset:** Same Ames Housing dataset used in L1 (`data/AmesHousing.csv`).  
**Goal:** Predict one of six sale-price categories.  
**Scope:** Setup, preprocessing, baseline, stratified/non-stratified splits, 5/10-fold cross-validation, and bias–variance discussion.

Run the notebook from the `L1ML` project root. All preprocessing is fitted **only on training data** to avoid leakage.

## 1. Environment setup
Run once in the VS Code terminal: `python -m pip install pandas numpy scikit-learn xgboost plotly optuna anywidget matplotlib seaborn ipykernel`.
Restart the notebook kernel after installation if necessary.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
print("XGBoost version:", xgb.__version__)

## 2. Load the original Ames Housing dataset
This is the **same dataset as L1**, not a substitute. The original CSV is tab-separated. If Person 1 has an already-cleaned version, use its exact path instead and check its delimiter.

In [ ]:
DATA_PATH = Path("data/AmesHousing.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing {DATA_PATH}. Run the notebook from the L1ML project root.")
housing = pd.read_csv(DATA_PATH, sep="\t")
print("Dataset shape:", housing.shape)
print("Missing values (top 10):")
print(housing.isna().sum().sort_values(ascending=False).head(10))
display(housing.head())

## 3. Six price classes
Intervals use **left-inclusive, right-exclusive** boundaries. E.g., a house priced exactly 100,000 belongs to class 1. `SalePrice` must be removed from the features: keeping it would leak the answer.

In [ ]:
BINS = [-np.inf, 100000, 130000, 180000, 250000, 400000, np.inf]
CLASS_NAMES = ["<100k", "100–130k", "130–180k", "180–250k", "250–400k", ">=400k"]
df = housing.dropna(subset=["SalePrice"]).copy()
df["PriceClass"] = pd.cut(df["SalePrice"], bins=BINS, labels=False, right=False).astype(int)
print(df["PriceClass"].value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES))))
print("Class shares (%):")
print((df["PriceClass"].value_counts(normalize=True).sort_index() * 100).round(2))
df["PriceClass"].value_counts().sort_index().plot.bar(title="Class distribution", xlabel="Price class (0–5)", ylabel="Number of houses")
plt.tight_layout(); plt.show()

## 4. Prepare features and leakage-safe preprocessing
- Numerical: median imputation.
- Categorical: most-frequent imputation, followed by one-hot encoding.
- The transformers are **inside the pipeline**, so cross-validation cannot learn category statistics from validation folds.
- `PID` is an identifier, not a meaningful predictive characteristic, and is excluded.
- Any engineered variable calculated using `SalePrice` must also be excluded.

In [ ]:
DROP_COLUMNS = ["SalePrice", "PriceClass", "PID"]
X = df.drop(columns=DROP_COLUMNS, errors="ignore")
y = df["PriceClass"]
numeric_columns = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()
print("Numerical features:", len(numeric_columns), "Categorical features:", len(categorical_columns))

def make_pipeline(n_estimators=150, max_depth=4, learning_rate=0.08):
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_pipe, numeric_columns),
        ("cat", categorical_pipe, categorical_columns)
    ])
    classifier = xgb.XGBClassifier(
        objective="multi:softprob", num_class=6,
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=learning_rate, subsample=0.9,
        colsample_bytree=0.9, tree_method="hist",
        eval_metric="mlogloss", random_state=RANDOM_STATE,
        n_jobs=2
    )
    return Pipeline([("preprocess", preprocess), ("classifier", classifier)])

## 5. Baseline: 20% test, with stratification
Stratification preserves approximately the same proportions of the six price classes in the training and test subsets. Evaluate **macro-F1** because it gives equal weight to each class, including rare expensive houses.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
baseline = make_pipeline()
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Macro-F1:", round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4))
print(classification_report(y_test, y_pred, labels=range(6), target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=range(6), display_labels=CLASS_NAMES, xticks_rotation=45, cmap="Blues")
plt.title("Baseline confusion matrix (stratified 20% test)")
plt.tight_layout(); plt.show()

## 6. Compare 10% and 20% test splits, with/without stratification
Every experiment uses the **same model configuration** and random seed. Non-stratified splits may yield unstable estimates for rare price classes. The table includes class counts in the test set to make this visible.

In [ ]:
split_results = []
for test_size in (0.10, 0.20):
    for stratify in (False, True):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_size, random_state=RANDOM_STATE,
            stratify=y if stratify else None
        )
        candidate = make_pipeline()
        candidate.fit(X_tr, y_tr)
        pred_tr = candidate.predict(X_tr)
        pred_te = candidate.predict(X_te)
        split_results.append({
            "test_size": test_size, "stratified": stratify,
            "n_train": len(y_tr), "n_test": len(y_te),
            "train_macro_f1": f1_score(y_tr, pred_tr, average="macro", zero_division=0),
            "test_accuracy": accuracy_score(y_te, pred_te),
            "test_macro_f1": f1_score(y_te, pred_te, average="macro", zero_division=0),
            "test_class_counts": str(y_te.value_counts().reindex(range(6), fill_value=0).tolist())
        })
split_table = pd.DataFrame(split_results)
display(split_table.round(4))

### Interpretation: split experiments
Compare **test_macro_f1** first, then accuracy. A 10% test split leaves more training data but gives a smaller, noisier test estimate. Stratification is especially relevant when some price ranges contain few examples. Do not automatically declare a strategy superior from a single random split; the next section uses repeated validation folds.

## 7. Cross-validation: k = 5 and k = 10
Compare ordinary `KFold` and `StratifiedKFold`, both with shuffling. Each fold trains a new pipeline from scratch. We report mean and standard deviation of macro-F1, and mean training macro-F1. Note: 10-fold CV trains ten models and can take longer.

In [ ]:
cv_results = []
for k in (5, 10):
    for stratified in (False, True):
        cv = (StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
              if stratified else KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE))
        scores = cross_validate(
            make_pipeline(), X, y, cv=cv,
            scoring={"macro_f1": "f1_macro", "accuracy": "accuracy"},
            return_train_score=True, n_jobs=1, error_score="raise"
        )
        cv_results.append({
            "k": k, "stratified": stratified,
            "train_macro_f1_mean": scores["train_macro_f1"].mean(),
            "validation_macro_f1_mean": scores["test_macro_f1"].mean(),
            "validation_macro_f1_std": scores["test_macro_f1"].std(),
            "validation_accuracy_mean": scores["test_accuracy"].mean(),
            "train_validation_gap": scores["train_macro_f1"].mean() - scores["test_macro_f1"].mean()
        })
cv_table = pd.DataFrame(cv_results)
display(cv_table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
labels = cv_table.apply(lambda r: f"k={int(r['k'])}, {'stratified' if r['stratified'] else 'plain'}", axis=1)
ax.bar(labels, cv_table["validation_macro_f1_mean"], yerr=cv_table["validation_macro_f1_std"], capsize=5)
ax.set_ylabel("Validation macro-F1 (mean ± std)")
ax.set_title("5-fold vs 10-fold cross-validation")
ax.set_ylim(0, 1)
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

## 8. Bias–variance tradeoff
**Bias** is systematic error due to a model that is too simple; **variance** reflects sensitivity to the training data. A large training–validation macro-F1 gap is a warning sign of overfitting, but **it does not directly measure statistical variance**. Cross-validation score standard deviation indicates how much performance changes across folds, though folds overlap and the estimate has limitations.

Compare model complexity using different tree depths on the **same stratified 5-fold CV**. Shallow trees can underfit; deeper trees may fit training data better but generalize worse.

In [ ]:
complexity_results = []
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for depth in (2, 4, 8):
    scores = cross_validate(
        make_pipeline(max_depth=depth), X, y, cv=cv5,
        scoring="f1_macro", return_train_score=True,
        n_jobs=1, error_score="raise"
    )
    complexity_results.append({
        "max_depth": depth,
        "train_macro_f1": scores["train_score"].mean(),
        "validation_macro_f1": scores["test_score"].mean(),
        "validation_std": scores["test_score"].std()
    })
complexity_table = pd.DataFrame(complexity_results)
display(complexity_table.round(4))
plt.figure(figsize=(8, 5))
plt.plot(complexity_table["max_depth"], complexity_table["train_macro_f1"], marker="o", label="Train macro-F1")
plt.errorbar(complexity_table["max_depth"], complexity_table["validation_macro_f1"], yerr=complexity_table["validation_std"], marker="o", capsize=4, label="CV macro-F1")
plt.xlabel("Maximum tree depth"); plt.ylabel("Macro-F1")
plt.title("Model complexity: training vs validation")
plt.legend(); plt.grid(alpha=0.25); plt.tight_layout(); plt.show()

## 9. Results and conclusions (complete after execution)
1. **Dataset and class balance:** Report the observed distribution of the six price classes. Are classes 0 or 5 rare?
2. **Baseline:** Copy test accuracy and macro-F1 from Section 5; explain any low per-class recall.
3. **Split experiments:** State which test fraction and stratification strategy achieved higher macro-F1. Consider sample-size effects and class imbalance.
4. **Cross-validation:** Compare mean macro-F1 and standard deviation for k=5 and k=10. Smaller standard deviation means more consistent fold scores, not necessarily a better classifier.
5. **Bias–variance:** Use Section 8 to discuss underfitting vs overfitting. Report the depth with the highest validation macro-F1, not the highest training score.

**Reproducibility:** Random state = 42; preprocessing occurs inside the pipeline; target price and identifier are excluded from predictors. The held-out test set should not be used for repeated hyperparameter tuning. Person 2 can use this baseline as the starting point for model evaluation and Optuna optimization.